In [24]:
import pandas as pd
from classes import StopNode

In [25]:
# resource ID for stop_times: 4e504fcf-c784-4a58-944a-14bb540dbbc7

stops = pd.read_csv('data/stops.csv')
stop_times = pd.read_csv('data/stop_times.csv')
trips = pd.read_csv('data/trips.csv')


In [ ]:
# randomly choosing one trip for each route
trips_lst = trips.groupby('route_id')['trip_id'].first()
trips_lst

route_id
12     2355010
14     1140010
18     9916070
19     8300020
1T     6852010
        ...   
O     13180020
P      1049020
U      1245020
V     11145040
W     13942180
Name: trip_id, Length: 123, dtype: int64

In [68]:
stops_test = pd.merge(stop_times, stops, how='left', on='stop_id')

# Keeping only those stops that correspond to our list of (randomly chosen) trips. Each route is represented once.
# Rows are ordered by trip_id then stop_sequence, so going down the rows and adding an edge between two successive stops within a trip will give all the right edges
stops_test = stops_test.loc[stops_test['trip_id'].isin(trips_lst), ['stop_id', 'stop_sequence', 'stop_name', 'trip_id', 'arrival_time', 'departure_time', 'stop_lat', 'stop_lon']]
stops_test.head()

,stop_id,stop_sequence,stop_name,trip_id,arrival_time,departure_time,stop_lat,stop_lon
1212,1594,1,Hayward BART,25020,10:56:00,10:56:00,37.669589,-122.086314
1213,1530,2,D St & Atherton St,25020,10:57:00,10:57:00,37.668785,-122.084227
1214,1442,3,A St & Flagg St,25020,10:59:24,10:59:24,37.669886,-122.092856
1215,5193,4,B St & Filbert St,25020,11:00:21,11:00:21,37.667426,-122.095865
1216,1849,5,Meekland Av & A St,25020,11:01:06,11:01:06,37.667028,-122.099046


In [73]:
# making graph representation (dicts for nodes and adjacency list for edges)

## nodes
node_subset = stops_test.groupby('stop_id').first()
#node_subset = node_subset.set_index('stop_id')
node_subset = node_subset.loc[:, ['stop_name', 'stop_lat', 'stop_lon']]
stops_test_dict = node_subset.to_dict('index')

# new syntax wow! dictionary comprehension. 
# {key:value for item in collection}
nodes_dict = {
    stop_id: StopNode(**value)
    for stop_id, value in stops_test_dict.items()
}

## edges
edges = stops_test.sort_values(['trip_id', 'stop_sequence'])
edges['next_stop_id'] = edges.groupby('trip_id')['stop_id'].shift(-1)
edges = edges.dropna(subset=['next_stop_id'])
edges['next_stop_id'] = edges['next_stop_id'].astype(int)
edges_dict = edges.groupby('stop_id')['next_stop_id'].apply(set).to_dict()

In [ ]:
edges_dict

3160